In [ ]:
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path

# Load Linux Libertine font
FONT_DIR = Path(__file__).parent / 'fonts' if '__file__' in dir() else Path('fonts')
for f in FONT_DIR.glob('*.otf'): 
    fontManager.addfont(str(f))

FONT_NAME = 'Linux Libertine O'
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set(
    context='paper',
    style='ticks',
    palette='deep',
    font=FONT_NAME,
    font_scale=1.8,
    rc={
        'mathtext.fontset': 'stix',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.labelweight': 'normal',
        'axes.titleweight': 'normal',
        'font.weight': 'normal',
    }
)
print(f'Font: {FONT_NAME}, Output: {OUTPUT_DIR}')

In [ ]:
# Load evaluation data
DATA_PATH = Path('../../data/eval_results/exports/aggregated/all_results.parquet')
df = pd.read_parquet(DATA_PATH)

# Clean dataset names (remove _semantic suffix)
df['dataset'] = df['dataset'].str.replace('_semantic', '', regex=False)

# Create retrieval_config column: method + hyde_mode
df['config'] = df.apply(
    lambda r: f"{r['method']}"
    + ('+HyDE' if r['hyde_mode'] == 'combined' else ''),
    axis=1
)

print(f'Loaded {len(df)} rows')
print(f'Datasets: {sorted(df["dataset"].unique())}')
print(f'Configs: {sorted(df["config"].unique())}')
df[['dataset', 'config', 'hit_at_1', 'hit_at_5', 'hit_at_10', 'mrr']].sort_values(['dataset', 'config'])

In [ ]:
# Hit@1 bar chart: 3 bars per dataset, UPO-aligned delta as stacked overlay
DATASET_ORDER = ['adventure_works', 'chembl', 'public_bi', 'fetaqa', 'fetaqapn', 'bird', 'chicago']
DATASET_LABELS = {
    'adventure_works': 'Ad.', 'bird': 'BD.', 'chembl': 'Ch.',
    'chicago': 'Cc.', 'fetaqa': 'FL.', 'fetaqapn': 'FM.',
    'public_bi': 'Pb.',
}
METHODS = ['bm25', 'vector', 'hybrid']
METHOD_LABELS = {'bm25': 'BM25', 'vector': 'BGE M3', 'hybrid': 'Hybrid'}

# Baseline (SOLO) performance per dataset (display order: Ad. Ch. Pb. FL. FM. BD. Cc.)
BASELINES = {
    'adventure_works': 0.789, 'chembl': 0.820, 'public_bi': 0.736,
    'fetaqa': 0.331, 'fetaqapn': 0.569, 'bird': 0.603,
    'chicago': 0.532,
}

# Refined Academic palette — teal blue + warm bronze + royal purple
COLORS = {
    'bm25':   ('#3C6E8F', '#8FB5CE'),   # teal blue
    'vector': ('#C08552', '#DFC1A0'),   # warm bronze
    'hybrid': ('#6B5B95', '#ADA4C4'),   # royal purple
}

# Pneuma baselines per retrieval method (display order: Ad. Ch. Pb. FL. FM. BD. Cc.)
PNEUMA = {
    'bm25': {  # Pneuma Sparse
        'adventure_works': 0.624, 'chembl': 0.734, 'public_bi': 0.684,
        'fetaqa': 0.259, 'fetaqapn': 0.498, 'bird': 0.531, 'chicago': 0.466,
    },
    'vector': {  # Pneuma Dense
        'adventure_works': 0.719, 'chembl': 0.791, 'public_bi': 0.554,
        'fetaqa': 0.282, 'fetaqapn': 0.396, 'bird': 0.539, 'chicago': 0.454,
    },
    'hybrid': {  # Pneuma Fusion
        'adventure_works': 0.789, 'chembl': 0.820, 'public_bi': 0.736,
        'fetaqa': 0.331, 'fetaqapn': 0.569, 'bird': 0.603, 'chicago': 0.532,
    },
}

# Pivot data
raw = df[df['hyde_mode'] == 'raw'].set_index(['dataset', 'method'])['hit_at_1']
hyde = df[df['hyde_mode'] == 'combined'].set_index(['dataset', 'method'])['hit_at_1']
delta = hyde - raw

n_datasets = len(DATASET_ORDER)
n_methods = len(METHODS)
bar_width = 0.22
x = np.arange(n_datasets) * 0.85  # compressed spacing

fig, ax = plt.subplots(figsize=(5.0, 3.0))

for i, method in enumerate(METHODS):
    positions = x + (i - 1) * bar_width
    base_vals = [raw.get((d, method), 0) for d in DATASET_ORDER]
    delta_vals = [delta.get((d, method), 0) for d in DATASET_ORDER]

    base_color, light_color = COLORS[method]

    # Base bars
    ax.bar(positions, base_vals, bar_width,
           color=base_color, edgecolor='white', linewidth=0.5,
           label=METHOD_LABELS[method], zorder=3)

    # UPO-aligned delta: stacked on top (positive) or hanging below (negative)
    for j, (pos, base, d) in enumerate(zip(positions, base_vals, delta_vals)):
        if abs(d) < 0.001:
            continue
        if d > 0:
            ax.bar(pos, d, bar_width, bottom=base,
                   facecolor='none', edgecolor=base_color, linewidth=0.6,
                   hatch='///', zorder=4)
        else:
            ax.bar(pos, abs(d), bar_width, bottom=base - abs(d),
                   facecolor='#E0E0E0', edgecolor='#777777', linewidth=0.6,
                   hatch='\\\\\\', alpha=0.85, zorder=4)

    # Pneuma baseline markers per bar (diamond markers)
    for j, ds in enumerate(DATASET_ORDER):
        bl = PNEUMA[method][ds]
        pos = positions[j]
        ax.scatter(pos, bl, marker='D', s=14, color='#555555',
                   edgecolors='white', linewidths=0.4, zorder=6)

# Vertical dividers between dataset groups
for idx in range(n_datasets - 1):
    mid = (x[idx] + x[idx + 1]) / 2
    ax.axvline(mid, color='#BBBBBB', linewidth=0.5, linestyle='-', zorder=1)

# Legend entries
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
upo_up_patch = Patch(facecolor='none', edgecolor='#888888', hatch='///', label='+UPO ↑')
upo_down_patch = Patch(facecolor='#E0E0E0', edgecolor='#777777', hatch='\\\\\\', label='+UPO ↓')
baseline_marker = Line2D([0], [0], marker='D', color='w', markerfacecolor='#555555',
                         markeredgecolor='white', markersize=5, label='Pneuma', linestyle='None')

ax.set_xticks(x)
ax.set_xticklabels([DATASET_LABELS[d] for d in DATASET_ORDER], fontsize=10)
ax.set_ylim(0.2, 0.95)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
ax.set_ylabel('Hit@1', fontsize=10)
ax.tick_params(axis='y', labelsize=10)

# Equal left/right margins
margin = bar_width * 1.5 + 0.08
ax.set_xlim(x[0] - margin, x[-1] + margin)

# Legend: explicit handle order — Row1: BM25, BGE M3, Hybrid; Row2: +UPO↑, +UPO↓, Pneuma
# matplotlib fills column-major with ncol, so interleave rows
method_handles = {lbl: h for h, lbl in zip(*ax.get_legend_handles_labels())}
ordered_handles = [
    method_handles['BM25'], upo_up_patch,       # col 0
    method_handles['BGE M3'], upo_down_patch,    # col 1
    method_handles['Hybrid'], baseline_marker,      # col 2
]
ax.legend(handles=ordered_handles, ncol=3, loc='upper right',
          frameon=True, fontsize=9,
          handlelength=1.5, handletextpad=0.4, columnspacing=0.8,
          fancybox=False, edgecolor='#AAAAAA', facecolor='white',
          framealpha=0.9)

ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'retrieval_hit1.pdf', bbox_inches='tight', pad_inches=0.02, dpi=300)
plt.show()
print(f'Saved to {OUTPUT_DIR / "retrieval_hit1.pdf"}')


# average improvement over BM25 (delta) per method, averaged across datasets
avg_delta = {method: np.mean([delta.get((d, method), 0) for d in DATASET_ORDER]) for method in METHODS}
print('Average Hit@1 improvement over BM25 (delta), averaged across datasets:')
for method, avg in avg_delta.items():
    print(f'  {METHOD_LABELS[method]}: {avg:.4f}')
avg_ds = {method: np.mean([delta.get((d, method), 0) for d in DATASET_ORDER]) for method in METHODS}

#relative improvement over BM25 (delta / BM25) per method, averaged across datasets
rel_avg_delta = {method: np.mean([
    delta.get((d, method), 0) / raw.get((d, 'bm25'), 1e-6) for d in DATASET_ORDER
]) for method in METHODS}
print('Average relative Hit@1 improvement over BM25 (delta / BM25), averaged across datasets:')
for method, avg in rel_avg_delta.items():
    print(f'  {METHOD_LABELS[method]}: {avg:.4f}')

